In [ ]:
## dataset
import os
# ===================== 配置中国镜像 =====================
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
import sys
# 1. 获取项目根目录（当前目录的父目录）
project_root = "/home/feng1/disk/backdoor"
# 2. 将根目录加入系统路径
sys.path.append(project_root)
os.chdir(project_root)

In [ ]:
import os
import pandas as pd
import torch
import numpy as np
from tqdm import tqdm
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

class CLIPRunner:
    """
    CLIP 模型管理类：
    - 自动下载并缓存指定目录
    - 从本地加载模型
    - 计算图片与文本的相似度
    """
    def __init__(self, model_name='openai/clip-vit-base-patch32', cache_dir='./clip_cache', device=None):
        self.model_name = model_name
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)

        if device:
            self.device = torch.device(device)
        else:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(f"正在加载模型到设备: {self.device} ...")
        self.model = CLIPModel.from_pretrained(
            model_name,
            cache_dir=cache_dir,
            use_safetensors=True
        ).to(self.device)
        self.processor = CLIPProcessor.from_pretrained(model_name, cache_dir=cache_dir)

    def load_image(self, image_path):
        try:
            image = Image.open(image_path).convert("RGB")
            return image
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
            return None

    def compute_similarity(self, images, texts):
        valid_images = [img for img in images if img is not None]
        if not valid_images:
            return None
            
        inputs = self.processor(text=texts, images=valid_images, return_tensors="pt", padding=True).to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs)
            image_embeds = outputs.image_embeds 
            text_embeds = outputs.text_embeds    

        image_embeds = image_embeds / image_embeds.norm(p=2, dim=-1, keepdim=True)
        text_embeds = text_embeds / text_embeds.norm(p=2, dim=-1, keepdim=True)

        similarity = image_embeds @ text_embeds.t()
        return similarity

    def predict(self, image_path, texts):
        sim = self.compute_similarity([image_path], texts)
        if sim is None:
            return -1, None
        best_idx = sim.argmax(dim=-1).item()
        return best_idx, sim.cpu().numpy()

print("CLIPRunner 类定义完成。")

In [ ]:
def test_single_image(clip_runner):
    """
    场景1: 单张图片基础测试
    """
    print("\n" + "="*50)
    print("开始测试: 单张图片匹配")
    print("="*50)

    # 这里的路径请根据实际情况修改
    test_image = "./model/test/test/1000268201.jpg"
    
    # 如果测试图片不存在，创建一个假图片用于演示
    if not os.path.exists(test_image):
        print(f"警告: 测试图片 {test_image} 不存在，生成随机图片用于演示...")
        os.makedirs(os.path.dirname(test_image), exist_ok=True)
        Image.new('RGB', (224, 224), color = 'red').save(test_image)

    test_texts = [      
        "A CHILD IN A PINK DRESS IS CLIMBING UP A SET OF STAIRS IN AN ENTRY WAY.",
        "A LITTLE GIRL IN A PINK DRESS GOING INTO A WOODEN CABIN.",
        "A LITTLE GIRL CLIMBING THE STAIRS TO HER PLAYHOUSE.",
        "A LITTLE GIRL CLIMBING INTO A WOODEN PLAYHOUSE.",
        "A GIRL GOING INTO A WOODEN BUILDING."
    ]

    best_idx, sim_matrix = clip_runner.predict(test_image, test_texts)
    
    if best_idx != -1:
        print(f"图片路径: {test_image}")
        print(f"最匹配文本索引: {best_idx}")
        print(f"最匹配文本: {test_texts[best_idx]}")
        print(f"相似度分数: {sim_matrix[0][best_idx]:.4f}")
        print("测试通过！")

# ==========================================
# 执行部分
# ==========================================
# 初始化 Runner (如果内存中已有实例可注释掉下一行)
clip_runner = CLIPRunner(cache_dir="./model/clip/clip_cache")

# 运行测试
test_single_image(clip_runner)